# Tech Challenge - State of Data Brasil (Fase 3)
### Camada Gold - Perguntas 1 a 7

Le a camada Silver unificada direto do S3.

- Fonte: `s3://tc-fase3-state-of-data-seu-id/silver_final/state_of_data_unificado/`
- Tabela equivalente no Athena: `db_tc_fase3.state_of_data_unificado`
- 14.005 linhas, 63 colunas mais a particao `ano_pesquisa`

**Nao trocar a fonte por CSV local.** A base no S3 e a unica versao
padronizada. Se cada pessoa rodar com um arquivo da propria maquina, o
grupo acaba comparando resultados de bases diferentes.

Credencial: no Learner Lab ela e temporaria e tem tres partes, incluindo
o `aws_session_token`. Pegue em AWS Details, Show, e cole no
`~/.aws/credentials`. Expira ao encerrar a sessao do lab.


## Configuração e dependências

In [ ]:
# ===================== INSTALACAO DE DEPENDENCIAS =====================
%pip install pyspark pandas matplotlib seaborn numpy ipykernel pyarrow s3fs


In [ ]:
# ===================== IMPORTS =====================
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import countDistinct

sns.set_theme(style="whitegrid")


In [ ]:
# ===================== CONFIGURACAO =====================
# Fonte unica de verdade: a camada Silver unificada no S3.
# Nao usar CSV local. Se cada um rodar com um arquivo da propria maquina,
# o grupo acaba analisando bases diferentes.
import os
import pandas as pd

BUCKET = "tc-fase3-state-of-data-seu-id"

CAMINHO_SILVER = f"s3://{BUCKET}/silver_final/state_of_data_unificado/"

# Saida da Gold. As imagens (savefig) precisam de caminho local, entao
# PASTA_SAIDA_GOLD e local. Os dados agregados podem ir pro S3 depois,
# com: aws s3 cp Gold_output/ s3://{BUCKET}/gold/ --recursive
PASTA_SAIDA_GOLD = "Gold_output"
CAMINHO_GOLD_S3 = f"s3://{BUCKET}/gold/"
OUT = PASTA_SAIDA_GOLD

os.makedirs(PASTA_SAIDA_GOLD, exist_ok=True)

print("Silver:", CAMINHO_SILVER)
print("Gold local:", os.path.abspath(PASTA_SAIDA_GOLD))


In [ ]:
# ===================== SPARK SESSION E CARGA DA SILVER =====================
os.environ["PYSPARK_PYTHON"] = "python3"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TechChallenge-Gold")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# A leitura do S3 e feita pelo pandas com s3fs, e nao pelo Spark.
# Motivo: o Spark local precisaria dos jars hadoop-aws e aws-java-sdk
# pra falar s3a, o que da trabalho de configurar. Sao 14 mil linhas,
# entao carregar em pandas e converter pro Spark e instantaneo.
_pdf_silver = pd.read_parquet(CAMINHO_SILVER)

# a particao ano_pesquisa volta como category, o Spark nao aceita
for _c in _pdf_silver.columns:
    if str(_pdf_silver[_c].dtype) == "category":
        _pdf_silver[_c] = _pdf_silver[_c].astype(str)


def carregar_silver():
    """Devolve a Silver unificada como DataFrame do Spark."""
    return spark.createDataFrame(_pdf_silver)


print("Silver carregada:", _pdf_silver.shape[0], "linhas,",
      _pdf_silver.shape[1], "colunas")


## Parte 1 – Perguntas 1 a 3

# Import e carga spark

In [ ]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dados = carregar_silver()
dados.createOrReplaceTempView("pesquisa_unificada")


In [ ]:
print("Num de colunas:", len(dados.columns))
print("Num de linhas:", dados.count())


# Padronização graficos

### GRÁFICO DE BARRAS

In [ ]:
# ============================================================
# PADRONIZAÇÃO DOS GRÁFICOS
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")

# Paleta personalizada
cores_genero = {
    "feminino": "#e75480",
    "masculino": "#3b4cc0",
    "outro": "#7e03a8",
    "nao_informado": "#350000"
}

# ============================================================
# 1) GRÁFICO DE BARRAS SIMPLES PADRONIZADO
# ============================================================

def plot_bar(df, x, y, title, xlabel="", ylabel="", figsize=(12,6), palette="viridis"):
    # Aceita Spark ou Pandas
    if hasattr(df, "toPandas"):
        df = df.toPandas()

    # Ordena
    df = df.sort_values(y, ascending=False)

    # Plot
    plt.figure(figsize=figsize)
    ax = sns.barplot(data=df, x=x, y=y, palette=palette)

    # Números nas barras
    for p in ax.patches:
        ax.annotate(
            format(p.get_height(), ".0f"),
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold",
            color="#333"
        )

    # Estética
    plt.title(title, fontsize=18, weight="bold")
    plt.xlabel(xlabel, fontsize=14)
    plt.ylabel(ylabel, fontsize=14)
    plt.xticks(rotation=45, ha="right", fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()
    plt.show()

# ============================================================
# 2) GRÁFICO DE BARRAS HORIZONTAIS PADRONIZADO
# ============================================================

def plot_barth(df, x, y, title, xlabel="", ylabel="", figsize=(12,6), palette="viridis"):
    # Aceita Spark ou Pandas
    if hasattr(df, "toPandas"):
        df = df.toPandas()

    # Ordena (horizontal → ordena pelo eixo Y, mas mantém X como categoria)
    df = df.sort_values(y, ascending=True)

    # Plot
    plt.figure(figsize=figsize)
    ax = sns.barplot(data=df, x=y, y=x, palette=palette)

    # Números nas barras
    for p in ax.patches:
        ax.annotate(
            format(p.get_width(), ".0f"),
            (p.get_width(), p.get_y() + p.get_height() / 2),
            ha="left",
            va="center",
            fontsize=12,
            fontweight="bold",
            color="#333"
        )

    # Estética
    plt.title(title, fontsize=18, weight="bold")
    plt.xlabel(xlabel if xlabel else y, fontsize=14)
    plt.ylabel(ylabel if ylabel else x, fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()
    plt.show()


### GRÁFICO DE DIVERSIDADE DE GÊNERO

In [ ]:
# ============================================================
# 2) GRÁFICO DE DIVERSIDADE DE GÊNERO POR CARGO (PADRONIZADO)
# ============================================================

def plot_genero_por_cargo(df, titulo="Distribuição Percentual de Gênero por Cargo"):
    # Aceita Spark ou Pandas
    if hasattr(df, "toPandas"):
        df = df.toPandas()

    # Remover cargo "outros"
    df = df[df["cargo"] != "outros"]

    # Pivot: cargo × genero → percentual
    pivot = df.pivot(index="cargo", columns="genero", values="percentual").fillna(0)

    # Ordenar pelo percentual de mulheres
    if "feminino" in pivot.columns:
        pivot = pivot.sort_values("feminino", ascending=False)

    # Plot
    plt.figure(figsize=(22,12))
    ax = pivot.plot(
        kind="bar",
        figsize=(22,12),
        color=[cores_genero.get(col, "#999999") for col in pivot.columns]
    )

    # Título e eixos
    plt.title(titulo, fontsize=18, weight="bold")
    plt.ylabel("Porcentagem (%)", fontsize=14)
    plt.xlabel("Cargo", fontsize=14)
    plt.xticks(rotation=45, ha="right", fontsize=12)
    plt.yticks(fontsize=12)

    # Exibir valores nas barras
    for container in ax.containers:
        labels = [f"{v:.1f}%" for v in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=11)

    plt.tight_layout()
    plt.show()

# ============================================================
# GRÁFICO DE GÊNERO POR SENIORIDADE (EFEITO FUNIL)
# ============================================================

def plot_genero_por_senioridade(df, titulo="Distribuição de Gênero por Nível de Senioridade (Efeito Funil)"):
    # Aceita Spark ou Pandas
    if hasattr(df, "toPandas"):
        df = df.toPandas()

    # Pivot: senioridade × genero → percentual
    pivot = df.pivot(index="senioridade", columns="genero", values="percentual").fillna(0)

    # Ordenar níveis de senioridade (junior → pleno → senior → especialista)
    ordem = ["junior", "pleno", "senior", "especialista"]
    pivot = pivot.reindex(ordem)

    # Plot horizontal empilhado
    plt.figure(figsize=(16,8))

    ax = pivot.plot(
        kind="barh",
        stacked=True,
        figsize=(16,8),
        color=[cores_genero.get(col, "#999999") for col in pivot.columns]
    )
    ax.legend(
            title="Gênero",
            loc="upper right",
            bbox_to_anchor=(1.15, 1.05),   # ajusta posição
            fontsize=12,
            title_fontsize=12
        )
    # Título e eixos
    plt.title(titulo, fontsize=18, weight="bold")
    plt.xlabel("Percentual (%)", fontsize=14)
    plt.ylabel("Senioridade", fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # Labels nas barras (somente feminino e masculino)
    for i, (idx, row) in enumerate(pivot.iterrows()):
        acumulado = 0
        for genero in pivot.columns:
            valor = row[genero]
            if valor > 0:
                ax.text(
                    acumulado + valor/2,
                    i,
                    f"{valor:.1f}%",
                    ha="center",
                    va="center",
                    fontsize=11,
                    color="white" if genero != "feminino" else "black",
                    fontweight="bold"
                )
            acumulado += valor

    plt.tight_layout()
    plt.show()



# 1 Como está estruturado o mercado brasileiro de dados?

## Cargo

In [ ]:
total_dados = dados.count()
gold_mercado_cargo = (
    dados.filter((F.col("cargo") != "nao_informado") & (F.col("cargo").isNotNull()))
    .groupBy("cargo")
    .agg(
        F.count("*").alias("quantidade")
    )
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade") / F.lit(total_dados) * 100,
            2
        )
    )
    .orderBy(F.desc("quantidade"))
)

gold_mercado_cargo.show(30, False)

In [ ]:
plot_bar(gold_mercado_cargo, "cargo", "quantidade", "Distribuição por Cargo", "Cargo", "Quantidade")

## Setor

In [ ]:
gold_mercado_setor = (
    dados.filter((F.col("setor") != "nao_informado") & (F.col("setor") != "outra_opcao"))
    .groupBy("setor")
    .agg(F.count("*").alias("qtde"))
)

gold_mercado_setor.show(40, False)

In [ ]:
plot_barth(gold_mercado_setor, "setor", "qtde", "Distribuição por Setor")

## Situação de trabalho

In [ ]:
# Dicionário Situação
map_situacao = {
    "empregado_clt": "CLT",
    "empreendedor_ou_empregado_cnpj": "CNPJ / Empreendedor",
    "estagiario": "Estagiário",
    "vivo_no_brasil_e_trabalho_remoto_para_empresa_de_fora_do_brasil": "Remoto p/ Exterior",
    "servidor_publico": "Servidor Público",
    "freelancer": "Freelancer",
    "prefiro_nao_informar": "Não Informou",
    "desempregado_e_nao_estou_buscando_recolocacao": "Desocupado",
    "vivo_fora_do_brasil_e_trabalho_para_empresa_de_fora_do_brasil": "Exterior",
    "somente_estudante_pos_graduacao": "Estudante (Pós)",
    "somente_estudante_graduacao": "Estudante (Graduação)",
    "desempregado_buscando_recolocacao": "Desempregado (Buscando)",
    "trabalho_na_area_academica_pesquisador": "Pesquisador / Acadêmico",
}

# Converte o dicionário para expressão Spark
map_sit = F.create_map(
    [F.lit(x) for kv in map_situacao.items() for x in kv]
)

gold_mercado_situacao = (
    dados
    .groupBy("situacao_trabalho")
    .agg(F.count("*").alias("qtde"))
    .withColumn("situacao_trabalho", map_sit[F.col("situacao_trabalho")])
    .orderBy("qtde")
)

gold_mercado_situacao.show(30, False)

In [ ]:
plot_bar(gold_mercado_situacao, "situacao_trabalho", "qtde", "Situação de Trabalho")

# 2 Quais perfis profissionais são mais valorizados?

## Remuneração x Cargo

In [ ]:
gold_remuneracao_cargo = (
    dados
    .filter(F.col("faixa_salarial_label") != "Sem resposta")
    .groupBy("cargo", "faixa_salarial_label")
    .agg(F.count("*").alias("quantidade"))
)

gold_remuneracao_cargo.show(50, False)

In [ ]:
#Gerar gráfico
ordem_faixa = [
    "R$ 0 a R$ 1.000",
    "R$ 1.001 a R$ 2.000",
    "R$ 2.001 a R$ 3.000",
    "R$ 3.001 a R$ 4.000",
    "R$ 4.001 a R$ 6.000",
    "R$ 6.001 a R$ 8.000",
    "R$ 8.001 a R$ 12.000",
    "R$ 12.001 a R$ 16.000",
    "R$ 16.001 a R$ 20.000",
    "R$ 20.001 a R$ 25.000",
    "R$ 25.001 a R$ 30.000",
    "R$ 30.001 a R$ 40.000",
    "R$ 40.001 ou mais",
    "Sem resposta"
]

df = gold_remuneracao_cargo.toPandas()

df["faixa_salarial_label"] = pd.Categorical(
    df["faixa_salarial_label"],
    categories=ordem_faixa,
    ordered=True
)

pivot = df.pivot_table(
    index="cargo",
    columns="faixa_salarial_label",
    values="quantidade",
    fill_value=0
).sort_index(axis=1)

pivot = df.pivot_table(
    index="cargo",
    columns="faixa_salarial_label",
    values="quantidade",
    fill_value=0
)

plt.figure(figsize=(20,12))
sns.heatmap(
    pivot,
    cmap="viridis",
    annot=True,
    fmt=".0f"
)

plt.title("Heatmap de Distribuição Salarial por Cargo", fontsize=18, weight="bold")
plt.xlabel("Faixa Salarial", fontsize=14)
plt.ylabel("Cargo", fontsize=14)
plt.tight_layout()
plt.show()


## Remuneracao X Senioridade

In [ ]:
# 1) Remover faixas e cargos indesejados
dados_limpo = dados.filter(
    (F.col("faixa_salarial_label") != "Sem resposta") &
    (~F.col("cargo").isin(["nao_informado", "outros", "outro"]))
)

# 2) Contagem por cargo × senioridade × faixa salarial
gold_remuneracao_senioridade = (
    dados_limpo
    .groupBy("cargo", "senioridade", "faixa_salarial_label")
    .agg(F.count("*").alias("qtde"))
)

gold_remuneracao_senioridade.show(5)

In [ ]:
# 3) Moda da faixa salarial por cargo × senioridade
w = Window.partitionBy("cargo", "senioridade").orderBy(F.desc("qtde"))

gold_moda = (
    gold_remuneracao_senioridade
    .withColumn("rank", F.row_number().over(w))
    .filter("rank = 1")
    .select("cargo", "senioridade", "faixa_salarial_label", "qtde")
)

# Extrair faixa mínima e máxima
df_sal = (
    gold_moda
    .withColumn("faixa_min", F.regexp_extract("faixa_salarial_label", r"R\$ (\d+)", 1).cast("int"))
    .withColumn("faixa_max", F.regexp_extract("faixa_salarial_label", r"a R\$ (\d+)", 1).cast("int"))
)

# Criar faixa média
df_sal = df_sal.withColumn(
    "faixa_media",
    F.when(F.col("faixa_max").isNull(), F.col("faixa_min"))
     .otherwise((F.col("faixa_min") + F.col("faixa_max")) / 2)
)

# Média estimada por cargo × senioridade
df_media = (
    df_sal.groupBy("cargo", "senioridade")
          .agg(
              F.avg("faixa_media").alias("media_estimada")
          )
)

# Converter para Pandas
pdf = df_media.toPandas()
pdf_filtrado = pdf[
    ~pdf["cargo"].isin(["outros", "nao_informado", "outro"])
]



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ordenar cargos pelo maior salário do especialista
ordem_cargos = (
    pdf_filtrado[pdf_filtrado["senioridade"] == "especialista"]
    .sort_values("media_estimada", ascending=False)["cargo"]
    .tolist()
)

pdf_sorted = pdf_filtrado.copy()
pdf_sorted["cargo"] = pd.Categorical(pdf_sorted["cargo"], categories=ordem_cargos, ordered=True)
pdf_sorted = pdf_sorted.sort_values("cargo")

plt.figure(figsize=(22, 12))

ax = sns.barplot(
    data=pdf_sorted,
    x="cargo",
    y="media_estimada",
    hue="senioridade",
    hue_order=["junior", "pleno", "senior", "especialista"],
    palette=["#1f77b4", "#2ca02c", "#17becf", "#98df8a"],
    width=0.75
)

# Labels em K
for bars in ax.containers:
    labels = [f"{bar.get_height():.0f}K" for bar in bars]
    ax.bar_label(bars, labels=labels, padding=3, fontsize=11)

plt.xticks(rotation=45, ha="right", fontsize=12)
plt.yticks(fontsize=12)

plt.xlabel("Cargo / Especialidade", fontsize=14)
plt.ylabel("Faixa Salarial Média (R$)", fontsize=14)

plt.title(
    "Remuneração Média Estimada (R$) por Cargo e Senioridade",
    fontsize=20,
    weight="bold"
)

ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.legend(
    title="Senioridade",
    title_fontsize=13,
    fontsize=12,
    loc="upper right"
)

plt.tight_layout()
plt.show()


# 3 Qual o cenário de diversidade de gênero nas carreiras de dados?

## Diversidade

In [ ]:
gold_diversidade_genero = (
    dados
    .groupBy("genero")
    .agg(
        F.count("*").alias("quantidade")
    )
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade") / F.lit(total_dados) * 100,
            2
        )
    )
)

gold_diversidade_genero.show()

In [ ]:
plot_bar(gold_diversidade_genero, "genero", "quantidade", "Distribuição Geral de Gênero", palette=cores_genero)

## Genero x Cargo

In [ ]:
gold_diversidade_cargo = (
    dados
    .filter((F.col("genero") != "outro")&(F.col("genero") != "nao_informado"))
    .groupBy("cargo", "genero")
    .agg(F.count("*").alias("quantidade"))
)

totais = (
    gold_diversidade_cargo
    .groupBy("cargo")
    .agg(F.sum("quantidade").alias("total_cargo"))
)

gold_diversidade_cargo = (
    gold_diversidade_cargo
    .join(totais, on="cargo", how="left")
    .withColumn("percentual", F.round(F.col("quantidade") / F.col("total_cargo") * 100, 2))
)

gold_diversidade_cargo.show(15, False)

In [ ]:
plot_genero_por_cargo(gold_diversidade_cargo)


## Genero x Senerioridade


In [ ]:
gold_diversidade_senioridade = (
    dados
        .filter(F.col("senioridade")!="nao_informado" )
        .filter((F.col("genero") != "outro")&(F.col("genero") != "nao_informado") )
        .groupBy("senioridade", "genero")
        .agg(F.count("*").alias("total"))
)

gold_diversidade_senioridade.show()

In [ ]:
# Total por senioridade
totais = (
    gold_diversidade_senioridade
    .groupBy("senioridade")
    .agg(F.sum("total").alias("total_senioridade"))
)

# Adicionar percentual
gold_div_sen = (
    gold_diversidade_senioridade
    .join(totais, on="senioridade", how="left")
    .withColumn("percentual", F.round(F.col("total") / F.col("total_senioridade") * 100, 2))
)
gold_div_sen.show()

In [ ]:
plot_genero_por_senioridade(gold_div_sen)

# Estrutura para gravar no gold

In [ ]:
#Estrutura gold
'''
s3://PATH/gold/...

├── mercado/
│   ├── cargo/
│   ├── setor/
│   └── situacao/
│
├── remuneracao/
│   ├── cargo/
│   └── senioridade/
│
└── diversidade/
    ├── genero/
    ├── genero_cargo/
    └── genero_senioridade
'''

#### ✔ 1. Mercado
Informações sobre o mercado de trabalho, percepção, contexto e posicionamento.

mercado/cargo/ → distribuição de cargos, frequência, popularidade

mercado/setor/ → setores mais comuns, setores emergentes

mercado/situacao/ → situação profissional (empregado, desempregado, transição)




#### ✔ 2. Remuneração
Tudo relacionado a salários, faixas, senioridade e evolução.

remuneracao/cargo/ → média salarial por cargo

remuneracao/senioridade/ → média salarial por nível

#### ✔ 3. Diversidade
Informações de gênero

diversidade/genero/ → distribuição geral

diversidade/genero_cargo/ → gênero por cargo

diversidade/genero_senioridade/ → gênero por senioridade

## Parte 2 – Perguntas 4 e 5

# 1.0 Config

In [ ]:
import sys
print(sys.executable)

In [ ]:
# Dependências já instaladas na célula de configuração no topo do notebook.
# %pip install pyspark pandas matplotlib seaborn ipykernel


In [ ]:
import seaborn as sns
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

In [ ]:
# CAMINHO_SILVER e PASTA_SAIDA_GOLD definidos na configuracao global.


In [ ]:
# Base carregada uma unica vez com carregar_silver().


In [ ]:
# Mesma base da configuracao global, sem reler arquivo
df = carregar_silver()

print("Total de linhas:", df.count())


# 2.0 Dev

In [ ]:
# ---------------------------------------------------------------
# Correção da ordem da faixa salarial (a coluna faixa_salarial_ordem
# veio zerada/inutilizável -> 99 em todas as linhas). Criamos o
# mapeamento correto manualmente para permitir ordenação e ponto médio.
# ---------------------------------------------------------------

In [ ]:
# Criando um dicionário para mapear as faixas salariais para valores numéricos
#df.info() ==> equivalente a df.printSchema() no PySpark

resumo = []
for c, tipo in df.dtypes:
    total_nao_nulo = df.filter(F.col(c).isNotNull()).count()
    total_distintos = df.select(c).distinct().count()
    resumo.append((c, tipo, total_nao_nulo, total_distintos))

df_resumo = spark.createDataFrame(resumo, ["coluna", "tipo", "nao_nulos", "distintos"])
df_resumo.show(70, truncate=False)

In [ ]:
import sys
print(sys.version)

import pyspark
print(pyspark.__version__)

In [ ]:
# Contando o número de linhas e colunas do DataFrame
df.printSchema()
df.show(5)
print("Total de linhas:", df.count())

# 3.0 Perguntas

### 4 - Quais tecnologias apresentam maior adoção entre os profissionais?

In [ ]:
#from awsglue.context import GlueContext
#from awsglue.job import Job
#from pyspark.context import SparkContext
#from pyspark.sql import functions as F
#import sys
#from awsglue.utils import getResolvedOptions

#args = getResolvedOptions(sys.argv, ['JOB_NAME'])
#sc = SparkContext()
#glueContext = GlueContext(sc)
#spark = glueContext.spark_session
#job = Job(glueContext)
#job.init(args['JOB_NAME'], args)

# --- Leitura da Silver unificada ---
#df = glueContext.create_dynamic_frame.from_catalog(
 #   database="db_silver",
  #  table_name="s_data_unificada_v3"
#).toDF()

# Padroniza os flags sim/nao -> 1/0 para poder somar
#flags_linguagem = ["usa_sql", "usa_python", "usa_r", "usa_scala"]
#flags_cloud     = ["cloud_aws", "cloud_gcp", "cloud_azure"]
#flags_bi        = ["bi_powerbi", "bi_tableau", "bi_looker", "bi_metabase"]
#flags_banco     = ["banco_postgres", "banco_mysql", "banco_sqlserver",
 #                   "banco_bigquery", "banco_databricks", "banco_snowflake", "banco_mongodb"]

#todas_flags = flags_linguagem + flags_cloud + flags_bi + flags_banco

#df_flags = df
#for c in todas_flags:
#    df_flags = df_flags.withColumn(c, F.when(F.col(c) == "sim", 1).otherwise(0))

# --- Unpivot: transforma colunas em linhas (categoria, tecnologia, usou) ---
#def montar_stack(colunas, categoria):
#    expr_pairs = ", ".join([f"'{c}', {c}" for c in colunas])
#    n = len(colunas)
#    return df_flags.select(
#        "ano_pesquisa",
#        F.expr(f"stack({n}, {expr_pairs}) as (tecnologia, usou)")
#    ).withColumn("categoria", F.lit(categoria))

#df_long = (
#    montar_stack(flags_linguagem, "linguagem")
#    .unionByName(montar_stack(flags_cloud, "cloud"))
#    .unionByName(montar_stack(flags_bi, "bi"))
#    .unionByName(montar_stack(flags_banco, "banco_dados"))
#)

# --- Agregação: total e % de adoção por tecnologia e ano ---
#total_respondentes_ano = df.groupBy("ano_pesquisa").agg(F.count("*").alias("total_geral"))

#gold_tecnologias_adocao = (
#    df_long.groupBy("ano_pesquisa", "categoria", "tecnologia")
#    .agg(F.sum("usou").alias("total_usuarios"))
#    .join(total_respondentes_ano, on="ano_pesquisa")
#    .withColumn("pct_adocao", F.round(F.col("total_usuarios") / F.col("total_geral") * 100, 1))
#    .drop("total_geral")
#    .orderBy("ano_pesquisa", "categoria", F.desc("pct_adocao"))
#)

#gold_tecnologias_adocao.write.mode("overwrite").partitionBy("ano_pesquisa") \
#    .parquet("s3://SEU-BUCKET/gold/tecnologias_adocao/")

# --- Extra: linguagem/BI/cloud PREFERIDA (não é flag, é categórica direta) ---
#gold_tecnologia_preferida = (
#    df.groupBy("ano_pesquisa", "linguagem_preferida", "bi_preferida", "cloud_preferida")
#    .agg(F.count("*").alias("total"))
#    .orderBy("ano_pesquisa", F.desc("total"))
#)
#gold_tecnologia_preferida.write.mode("overwrite").partitionBy("ano_pesquisa") \
#    .parquet("s3://SEU-BUCKET/gold/tecnologia_preferida/")





In [ ]:
from pyspark.sql import functions as F

# Padroniza os flags sim/nao -> 1/0 para poder somar
flags_linguagem = ["usa_sql", "usa_python", "usa_r", "usa_scala"]
flags_cloud     = ["cloud_aws", "cloud_gcp", "cloud_azure"]
flags_bi        = ["bi_powerbi", "bi_tableau", "bi_looker", "bi_metabase"]
flags_banco     = ["banco_postgres", "banco_mysql", "banco_sqlserver",
                    "banco_bigquery", "banco_databricks", "banco_snowflake", "banco_mongodb"]

todas_flags = flags_linguagem + flags_cloud + flags_bi + flags_banco

df_flags = df
for c in todas_flags:
    df_flags = df_flags.withColumn(c, F.when(F.col(c) == "sim", 1).otherwise(0))

# --- Unpivot: transforma colunas em linhas (categoria, tecnologia, usou) ---
def montar_stack(colunas, categoria):
    expr_pairs = ", ".join([f"'{c}', {c}" for c in colunas])
    n = len(colunas)
    return df_flags.select(
        "ano_pesquisa",
        F.expr(f"stack({n}, {expr_pairs}) as (tecnologia, usou)")
    ).withColumn("categoria", F.lit(categoria))

df_long = (
    montar_stack(flags_linguagem, "linguagem")
    .unionByName(montar_stack(flags_cloud, "cloud"))
    .unionByName(montar_stack(flags_bi, "bi"))
    .unionByName(montar_stack(flags_banco, "banco_dados"))
)

# --- Agregação: total e % de adoção por tecnologia e ano ---
total_respondentes_ano = df.groupBy("ano_pesquisa").agg(F.count("*").alias("total_geral"))

gold_tecnologias_adocao = (
    df_long.groupBy("ano_pesquisa", "categoria", "tecnologia")
    .agg(F.sum("usou").alias("total_usuarios"))
    .join(total_respondentes_ano, on="ano_pesquisa")
    .withColumn("pct_adocao", F.round(F.col("total_usuarios") / F.col("total_geral") * 100, 1))
    .drop("total_geral")
    .orderBy("ano_pesquisa", "categoria", F.desc("pct_adocao"))
)

gold_tecnologias_adocao.show(50, truncate=False)

# Salva local (depois você troca por s3://... quando for pro Glue Job na AWS)
# Em vez de gold_tecnologias_adocao.write.parquet(...)
gold_tecnologias_adocao.toPandas().to_csv(f"{PASTA_SAIDA_GOLD}/tecnologias_adocao.csv", index=False)


## 3.1 Graficos 4

### Categorias (Ano mais recente 2025-2026)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Converte para pandas (tabela pequena, sem problema)
df_tech_pd = gold_tecnologias_adocao.toPandas()

# Foca no ano mais recente para a visão executiva
ano_foco = "2025_2026"
df_plot = df_tech_pd[df_tech_pd["ano_pesquisa"] == ano_foco]

categorias = df_plot["categoria"].unique()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, cat in enumerate(categorias):
    dados_cat = df_plot[df_plot["categoria"] == cat].sort_values("pct_adocao", ascending=True)
    axes[i].barh(dados_cat["tecnologia"], dados_cat["pct_adocao"], color=sns.color_palette("viridis", len(dados_cat)))
    axes[i].set_title(f"Adoção de {cat.replace('_', ' ').title()} ({ano_foco})", fontsize=12, fontweight="bold")
    axes[i].set_xlabel("% de adoção")
    for j, v in enumerate(dados_cat["pct_adocao"]):
        axes[i].text(v + 0.5, j, f"{v}%", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(f"{PASTA_SAIDA_GOLD}/grafico_tecnologias_adocao.png", dpi=200, bbox_inches="tight")
plt.show()


Gráfico 1 — Categorias (Ano mais recente 2025-2026)

A visão consolidada por categoria em 2025/2026 confirma o "tripé" de competências mais demandado no mercado brasileiro de dados: SQL (50,5%) e Python (55,2%) lideram entre as linguagens, Power BI (35,6%) domina entre as ferramentas de visualização, e Postgres (22,1%) aparece como o banco de dados mais adotado. Do lado de cloud, nenhum provedor ultrapassa 20% de adoção isolada, o que sugere um mercado ainda fragmentado entre AWS, GCP e Azure, sem um "vencedor" absoluto como acontece com linguagens e BI.

Insight-chave: "O mercado converge fortemente em linguagens e BI, mas permanece fragmentado em cloud."

### Ranking simples do ano mais recente

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
df_tech_pd = gold_tecnologias_adocao.toPandas()

ano_foco = "2025_2026"
df_plot = df_tech_pd[
    (df_tech_pd["ano_pesquisa"] == ano_foco) & (df_tech_pd["categoria"] == "linguagem")
].sort_values("pct_adocao", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
cores = ["#d9d9d9"] * len(df_plot)
cores[-1] = "#2E86AB"  # destaca só a tecnologia líder

ax.barh(df_plot["tecnologia"], df_plot["pct_adocao"], color=cores)
ax.set_title("Linguagens mais utilizadas em 2025/2026", fontsize=13, fontweight="bold")
ax.set_xlabel("% de profissionais que utilizam")

for i, v in enumerate(df_plot["pct_adocao"]):
    ax.text(v + 0.5, i, f"{v}%", va="center", fontsize=10)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig(f"{PASTA_SAIDA_GOLD}/grafico_linguagens_ranking.png", dpi=200, bbox_inches="tight")
plt.show()


 Gráfico 2 — Linguagens mais utilizadas em 2025/2026
No recorte mais recente, Python lidera com 55,2% de adoção, seguido de perto por SQL (50,5%) — uma inversão em relação aos anos anteriores, quando SQL era a líder isolada. R aparece distante, com 9,4%, e Scala é marginal, com apenas 2,0%.

Insight-chave: "Pela primeira vez no período analisado, Python ultrapassa o SQL como linguagem mais utilizada."





### 2 tecnologias no tempo (evolução com foco)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
df_tech_pd = gold_tecnologias_adocao.toPandas()

tecnologias_foco = ["usa_python", "usa_sql"]  # ajuste para o que quiser destacar
df_foco = df_tech_pd[
    (df_tech_pd["categoria"] == "linguagem") & (df_tech_pd["tecnologia"].isin(tecnologias_foco))
]

fig, ax = plt.subplots(figsize=(8, 5))
cores = {"usa_python": "#2E86AB", "usa_sql": "#F24236"}

for tec in tecnologias_foco:
    dados = df_foco[df_foco["tecnologia"] == tec]
    ax.plot(dados["ano_pesquisa"], dados["pct_adocao"], marker="o", linewidth=3,
            color=cores[tec], label=tec.replace("usa_", "").upper())
    # rótulo direto no fim da linha, sem legenda
    ax.text(2.05, dados["pct_adocao"].iloc[-1], tec.replace("usa_", "").upper(),
            fontsize=11, fontweight="bold", color=cores[tec], va="center")

ax.set_title("Python segue como linguagem dominante entre 2023-2026", fontsize=13, fontweight="bold")
ax.set_ylabel("% de adoção")
ax.set_xlim(-0.3, 2.5)
sns.despine()
plt.tight_layout()
plt.savefig(f"{PASTA_SAIDA_GOLD}/grafico_evolucao_foco.png", dpi=200, bbox_inches="tight")
plt.show()


Gráfico 3 — 2 tecnologias no tempo (evolução com foco: Python x SQL)

Ao longo dos três ciclos da pesquisa, o comportamento das duas linguagens diverge: o SQL parte na liderança (59,6% em 2023/2024), mas cai de forma constante até 50,5% em 2025/2026. O Python, por outro lado, cresce entre o primeiro e o segundo ano (53,4% → 56,3%) e se estabiliza próximo desse patamar (55,2%) no ciclo mais recente — resultando na inversão de liderança observada no último ano.

Insight-chave: "A trajetória de queda do SQL e a estabilidade do Python culminam na troca de liderança entre 2024/2025 e 2025/2026."

### Comparação Direta (Inicio - Fim)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")
df_tech_pd = gold_tecnologias_adocao.toPandas()

categoria_foco = "linguagem"  # troque para "cloud", "bi" ou "banco_dados" se quiser
anos_comparar = ["2023_2024", "2025_2026"]

df_comp = df_tech_pd[
    (df_tech_pd["categoria"] == categoria_foco) & (df_tech_pd["ano_pesquisa"].isin(anos_comparar))
]

# Pivota para ter uma linha por tecnologia, colunas = anos
df_pivot = df_comp.pivot(index="tecnologia", columns="ano_pesquisa", values="pct_adocao").fillna(0)
df_pivot = df_pivot.sort_values(anos_comparar[-1], ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(df_pivot))
altura = 0.35

ax.barh(y - altura/2, df_pivot[anos_comparar[0]], height=altura, label="2023/2024", color="#B0B0B0")
ax.barh(y + altura/2, df_pivot[anos_comparar[1]], height=altura, label="2025/2026", color="#2E86AB")

ax.set_yticks(y)
ax.set_yticklabels(df_pivot.index.str.replace("usa_", "").str.upper())
ax.set_xlabel("% de adoção")
ax.set_title("Comparativo de adoção: 2023/2024 vs 2025/2026", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")

# Rótulo com a variação (delta) ao lado de cada par de barras
for i, tec in enumerate(df_pivot.index):
    delta = df_pivot[anos_comparar[1]].iloc[i] - df_pivot[anos_comparar[0]].iloc[i]
    sinal = "+" if delta >= 0 else ""
    cor_delta = "#2E7D32" if delta >= 0 else "#C62828"
    ax.text(max(df_pivot.loc[tec]) + 2, i, f"{sinal}{delta:.1f}p.p.", va="center",
            fontsize=9, fontweight="bold", color=cor_delta)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig(f"{PASTA_SAIDA_GOLD}/grafico_comparativo_2023_2025.png", dpi=200, bbox_inches="tight")
plt.show()


Grafico 4 - O que esse gráfico entrega de diferente das outras opções:

Compara diretamente início vs fim do período (2023/2024 → 2025/2026), sem o "ruído" do ano do meio
Barras pareadas por tecnologia (fácil ver lado a lado)
Rótulo de variação em pontos percentuais (+8.2p.p. ou -3.1p.p.) ao lado de cada par — isso é o que realmente conta a história pro executivo: não só "quanto é hoje", mas "quanto mudou"
Cor verde/vermelho no delta reforça visualmente crescimento vs queda

Essa costuma ser a favorita em apresentações de negócio porque responde duas perguntas ao mesmo tempo: "onde estamos" e "para onde estamos indo" — sem precisar de mais de um gráfico.

Aqui já revela um insight interessante: SQL caiu 9.1 p.p. enquanto Python subiu 1.8 p.p., sinal de que Python está ganhando espaço como a linguagem mais "estável"/dominante entre 2023-2026. Isso já é uma frase pronta pro seu Storytelling.

### Panorama de adoção de tecnologias (2023-2026)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("white")

# Traz a tabela gold para pandas
df_tech_pd = gold_tecnologias_adocao.toPandas()

# Limpa os nomes das tecnologias para exibição
df_tech_pd["tecnologia_label"] = (
    df_tech_pd["tecnologia"]
    .str.replace("usa_", "", regex=False)
    .str.replace("cloud_", "", regex=False)
    .str.replace("bi_", "", regex=False)
    .str.replace("banco_", "", regex=False)
    .str.upper()
)

# Pivota: linhas = tecnologia, colunas = ano, valores = % adoção
df_pivot = df_tech_pd.pivot_table(index="tecnologia_label", columns="ano_pesquisa", values="pct_adocao")

# Ordena de forma decrescente geral, pelo ano mais recente
df_pivot = df_pivot.sort_values("2025_2026", ascending=False)
matriz = df_pivot[["2023_2024", "2024_2025", "2025_2026"]]

fig, ax = plt.subplots(figsize=(7, 11))
sns.heatmap(
    matriz,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    cbar_kws={"label": "% de adoção"},
    linewidths=0.5,
    linecolor="white",
    ax=ax
)
ax.set_title("Panorama de adoção de tecnologias (2023-2026)\nOrdenado por adoção em 2025/2026",
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticklabels(["2023/2024", "2024/2025", "2025/2026"], rotation=0)

plt.tight_layout()
plt.savefig(f"{PASTA_SAIDA_GOLD}/heatmap_panorama_tecnologias.png", dpi=200, bbox_inches="tight")
plt.show()


🗺️ Panorama de adoção de tecnologias (2023-2026)

Storytelling executivo (tom direto):

Três tecnologias dominam o mercado brasileiro de dados: SQL, Python e Power BI, com adoção acima de 35% em todos os anos analisados. Esse é o conjunto mínimo de competências que qualquer profissional da área precisa dominar hoje.

Um segundo grupo — Postgres, SQL Server, Databricks e AWS — oscila entre 15% e 30%, mostrando que bancos de dados e provedores cloud ainda são escolhas fragmentadas, sem um padrão único de mercado.

O dado mais relevante: a AWS caiu 12 pontos percentuais em três anos (31,0% → 19,0%), a maior queda entre todas as tecnologias monitoradas. Foi líder isolada em cloud e hoje disputa espaço com GCP e Azure.

Tecnologias de nicho — Scala, Snowflake e Metabase — seguem com adoção abaixo de 6%, fora do radar prioritário da maioria dos profissionais.

Frase de impacto para o slide:

"3 tecnologias dominam o mercado. 1 está em queda livre: a AWS perdeu 12 pontos percentuais em 3 anos."

## Pergunta 5 - Qual é o índice de adoção de Inteligência Artificial e seu impacto?

In [ ]:
#Validando nome das colunas que contém "ia" (inteligência artificial) para análise futura.

colunas_ia = [c for c in df.columns if "ia" in c.lower()]
print(colunas_ia)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# --- 1) Índice geral de adoção de IA por ano ---
w_ano = Window.partitionBy("ano_pesquisa")

gold_ia_adocao_geral = (
    df.groupBy("ano_pesquisa", "usa_ia")
    .agg(F.count("*").alias("total"))
    .withColumn("total_ano", F.sum("total").over(w_ano))
    .withColumn("pct", F.round(F.col("total") / F.col("total_ano") * 100, 1))
    .drop("total_ano")
    .orderBy("ano_pesquisa", F.desc("total"))
)
gold_ia_adocao_geral.show(20, truncate=False)

# --- 2) Como a IA é usada no dia a dia (entre quem respondeu) ---
gold_ia_tipo_uso = (
    df.filter(F.col("uso_ia") != "nao_informado")
    .groupBy("ano_pesquisa", "uso_ia")
    .agg(F.count("*").alias("total"))
    .orderBy("ano_pesquisa", F.desc("total"))
)
gold_ia_tipo_uso.show(20, truncate=False)

# --- 3) Prioridade estratégica da IA nas empresas (proxy de impacto organizacional) ---
gold_ia_prioridade = (
    df.filter(F.col("ia_prioridade") != "nao_informado")
    .groupBy("ano_pesquisa", "ia_prioridade", "ia_prioridade_ordem")
    .agg(F.count("*").alias("total"))
    .orderBy("ano_pesquisa", "ia_prioridade_ordem")
)
gold_ia_prioridade.show(20, truncate=False)

# --- 4) Adoção de IA por senioridade e cargo (impacto no perfil profissional) ---
gold_ia_por_perfil = (
    df.filter(F.col("usa_ia") != "nao_informado")
    .groupBy("ano_pesquisa", "senioridade", "usa_ia")
    .agg(F.count("*").alias("total"))
    .orderBy("ano_pesquisa", "senioridade", F.desc("total"))
)
gold_ia_por_perfil.show(30, truncate=False)

# --- 5) Motivos de NÃO adoção (barreiras) — texto multi-resposta separado por ", " ---
df_motivos = (
    df.filter(F.col("motivos_nao_ia").isNotNull())
    .withColumn("motivo", F.explode(F.split(F.col("motivos_nao_ia"), ", ")))
    .withColumn("motivo", F.trim(F.col("motivo")))
)

gold_ia_motivos_nao_adocao = (
    df_motivos.groupBy("ano_pesquisa", "motivo")
    .agg(F.count("*").alias("total"))
    .orderBy("ano_pesquisa", F.desc("total"))
)
gold_ia_motivos_nao_adocao.show(30, truncate=False)

### 5.1 Adoção geral de IA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

df_ia_pd = gold_ia_adocao_geral.toPandas()
pivot1 = df_ia_pd.pivot(index="ano_pesquisa", columns="usa_ia", values="pct").fillna(0)
pivot1 = pivot1[["sim", "nao", "nao_informado"]]

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(pivot1))
ax.plot(x, pivot1["sim"], marker="o", linewidth=3, color="#2E86AB", label="Usa IA")
ax.fill_between(x, pivot1["sim"], alpha=0.15, color="#2E86AB")

for i, v in enumerate(pivot1["sim"]):
    ax.text(i, v + 2, f"{v}%", ha="center", fontweight="bold", fontsize=11, color="#2E86AB")

ax.set_xticks(x)
ax.set_xticklabels(["2023/2024", "2024/2025", "2025/2026"])
ax.set_ylim(0, 80)
ax.set_ylabel("% de profissionais que usam IA")
ax.set_title("Adoção de Inteligência Artificial entre profissionais de dados", fontsize=13, fontweight="bold")
sns.despine()
plt.tight_layout()
plt.savefig(f"{PASTA_SAIDA_GOLD}/grafico_ia_adocao_geral.png", dpi=200, bbox_inches="tight")
plt.show()


📈 Gráfico 1 — Adoção de Inteligência Artificial entre profissionais de dados

Storytelling executivo:

A adoção de IA entre profissionais de dados no Brasil é alta e consistente: 57,2% em 2023/2024, pico de 64,8% em 2024/2025, e 59,0% em 2025/2026. A leve queda no último ciclo não indica recuo real de uso — coincide com um salto expressivo no percentual de "não informado" (28,7% → 39,7%), sugerindo maior cautela ou sensibilidade dos respondentes ao divulgar o uso de IA, não menor adoção.

Insight-chave:

"Mais de 6 em cada 10 profissionais de dados já usam IA no trabalho — e o dado pode estar subestimado, dado o aumento de não-respostas em 2025/2026."

### 5.2 Como a IA é usada

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("white")

df_uso_pd = gold_ia_tipo_uso.toPandas()

ordem = ["gratuito", "codigo", "pago_proprio", "pago_empresa", "nao_usa"]
labels = {
    "gratuito": "Gratuita (ex: ChatGPT)", "codigo": "Copilot/código",
    "pago_proprio": "Paga (bolso próprio)", "pago_empresa": "Paga (empresa)", "nao_usa": "Não usa"
}
cores = {
    "gratuito": "#A8DADC", "codigo": "#457B9D", "pago_proprio": "#F4A261",
    "pago_empresa": "#2E86AB", "nao_usa": "#D9D9D9"
}

# Recalcula % dentro de cada ano (garante que soma 100%)
df_uso_pd["pct"] = df_uso_pd.groupby("ano_pesquisa")["total"].transform(lambda s: round(s / s.sum() * 100, 1))
pivot2 = df_uso_pd.pivot(index="ano_pesquisa", columns="uso_ia", values="pct").fillna(0)
pivot2 = pivot2[ordem]

fig, ax = plt.subplots(figsize=(9, 5.5))
bottom = [0] * len(pivot2)
x = range(len(pivot2))

for cat in ordem:
    vals = pivot2[cat].values
    ax.bar(x, vals, bottom=bottom, label=labels[cat], color=cores[cat], width=0.6)
    for i, v in enumerate(vals):
        if v > 4:
            cor_texto = "white" if cat in ["pago_empresa", "codigo"] else "#333"
            ax.text(i, bottom[i] + v/2, f"{v:.0f}%", ha="center", va="center",
                     fontsize=9, fontweight="bold", color=cor_texto)
    bottom = [b + v for b, v in zip(bottom, vals)]

ax.set_xticks(x)
ax.set_xticklabels(["2023/2024", "2024/2025", "2025/2026"])
ax.set_ylabel("% (entre quem respondeu)")
ax.set_title("Como os profissionais usam IA no dia a dia", fontsize=13, fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig(f"{PASTA_SAIDA_GOLD}/grafico_ia_tipo_uso.png", dpi=200, bbox_inches="tight")
plt.show()


📊 Gráfico 2 — Como os profissionais usam IA no dia a dia

Storytelling executivo (o mais forte dos dois):

A forma de uso de IA passou por uma transformação completa em três anos. Em 2023/2024, 59% usavam apenas ferramentas gratuitas (tipo ChatGPT) e só 6% tinham a IA paga pela empresa. Em 2025/2026, esse cenário se inverteu: 42% já usam soluções pagas custeadas pela empresa, enquanto o uso gratuito caiu para 22%. Uso via ferramentas de código (Copilot) se manteve estável, entre 8% e 15%. O percentual de quem não usa IA praticamente zerou, caindo de 19% para 1,9%.

Insight-chave:

"IA deixou de ser ferramenta pessoal e virou investimento institucional: em 3 anos, o uso pago pela empresa saltou de 6% para 42% — um crescimento de 7x."

Frase de impacto para o slide:

"De curiosidade individual a prioridade corporativa: as empresas assumiram o custo da IA."

analisar dados de copilot no grafico
ver a possibilidade de analise por tipo de IA


In [ ]:
gold_ia_prioridade = (
    df.filter(F.col("ia_prioridade") != "nao_informado")
    .groupBy("ano_pesquisa", "ia_prioridade", "ia_prioridade_ordem")
    .agg(F.count("*").alias("total"))
    .orderBy("ano_pesquisa", "ia_prioridade_ordem")
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("white")

df_prior_pd = gold_ia_prioridade.toPandas()

ordem = ["foco_principal", "prioritaria", "parcial", "nao_prioridade", "nao_sei"]
labels = {
    "foco_principal": "Foco principal", "prioritaria": "Prioritária", "parcial": "Parcial",
    "nao_prioridade": "Não é prioridade", "nao_sei": "Não sei"
}
cores = {
    "foco_principal": "#1D3557", "prioritaria": "#2E86AB", "parcial": "#8ECAE6",
    "nao_prioridade": "#D9D9D9", "nao_sei": "#F1C40F"
}

# Recalcula % dentro de cada ano (garante soma 100%)
df_prior_pd["pct"] = df_prior_pd.groupby("ano_pesquisa")["total"].transform(lambda s: round(s / s.sum() * 100, 1))
pivot = df_prior_pd.pivot(index="ano_pesquisa", columns="ia_prioridade", values="pct").fillna(0)
pivot = pivot[ordem]

fig, ax = plt.subplots(figsize=(9, 5.5))
bottom = [0] * len(pivot)
x = range(len(pivot))

for cat in ordem:
    vals = pivot[cat].values
    ax.bar(x, vals, bottom=bottom, label=labels[cat], color=cores[cat], width=0.6)
    for i, v in enumerate(vals):
        if v > 4:
            cor_texto = "white" if cat in ["foco_principal", "prioritaria"] else "#333"
            ax.text(i, bottom[i] + v/2, f"{v:.0f}%", ha="center", va="center",
                     fontsize=9, fontweight="bold", color=cor_texto)
    bottom = [b + v for b, v in zip(bottom, vals)]

ax.set_xticks(x)
ax.set_xticklabels(["2023/2024", "2024/2025", "2025/2026"])
ax.set_ylabel("% (entre quem respondeu)")
ax.set_title("Prioridade estratégica da IA nas empresas", fontsize=13, fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig(f"{PASTA_SAIDA_GOLD}/grafico_ia_prioridade.png", dpi=200, bbox_inches="tight")
plt.show()


📊 Prioridade estratégica da IA nas empresas
Contexto

Além de medir se o profissional usa IA e como ele a utiliza no dia a dia, a pesquisa perguntou qual o grau de prioridade estratégica que a empresa atribui à Inteligência Artificial. Essa métrica é o indicador mais direto de impacto organizacional dentro do levantamento — não mede uso individual, mas posicionamento institucional.

Achado principal

Houve uma reconfiguração completa na priorização de IA entre 2023/2024 e 2025/2026. A soma das categorias "Foco principal" + "Prioritária" — que representam empresas que tratam IA como aposta estratégica real — saltou de 36,2% para 60,6%, quase dobrando em três anos. Na direção oposta, "Não é prioridade" caiu de 28,8% para 11,3%, menos da metade do patamar inicial. A categoria "Parcial" se manteve relativamente estável (25%-31%), sugerindo que a migração aconteceu majoritariamente de "não prioridade" direto para "prioritária/foco principal" — não um avanço gradual passando por "parcial".

Implicação para o negócio

Esse é o dado que comprova, de fato, o impacto da IA mencionado na pergunta 5 — não basta saber que a IA é usada (índice de adoção), é preciso saber se ela mudou de status dentro das organizações. A resposta é sim: em três anos, a IA deixou de ser tema secundário e passou a ser prioridade absoluta ou majoritária para 6 em cada 10 empresas do setor de dados no Brasil. Combinado com o achado do gráfico anterior (uso pago pela empresa saltando de 6% para 42%), fica claro que prioridade estratégica e investimento financeiro caminham juntos — a empresa que prioriza IA é a mesma que paga por ela.

Recomendação
Para empresas que ainda classificam IA como "não prioridade" (hoje 11,3%, mas ainda relevante), o risco é ficar para trás de uma maioria de mercado que já mudou de postura — esse dado é argumento direto para elevar IA na pauta estratégica;
Para consultorias e áreas de RH, vale usar esse indicador como critério de maturidade digital do cliente/empresa — perguntar "qual a prioridade de IA?" é mais revelador do que perguntar apenas "vocês usam IA?";
Programas de capacitação e contratação devem se antecipar a essa tendência: se a prioridade estratégica já dobrou em três anos, a demanda por profissionais capazes de implementar e governar IA (não apenas usá-la) tende a crescer no mesmo ritmo.

Frase de abertura ou fechamento do bloco de IA (fechamento da pergunta 5 como um todo):

"Em três anos, a IA saiu da lista de 'não prioridade' de quase 1 em cada 3 empresas para se tornar aposta estratégica de 6 em cada 10 — adoção e impacto caminharam juntos, e o investimento financeiro comprova isso."

## Parte 3 – Perguntas 6 e 7

In [ ]:
# Dependências já instaladas na célula de configuração no topo do notebook.
# !pip install pyspark


#Instalação do PySpark

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import countDistinct
from pyspark.sql.window import Window

###Ajustes para prevenção de erros

In [ ]:
# A SparkSession global já foi criada com as mesmas configurações
# (spark.driver.memory=4g, spark.sql.shuffle.partitions=8) na seção de configuração no topo.


###Importação de Bases

In [ ]:
# Base carregada uma unica vez com carregar_silver().


In [ ]:
# Mesma base da configuracao global, sem reler arquivo
df = carregar_silver()


In [ ]:
print("Total de registros:", df.count())
print("Colunas:", len(df.columns))

#Resposta 6 - Existem diferenças relevantes entre regiões, senioridades ou modelos de trabalho?

##Distribuição por Quantidade - Região, Senioridade e Forma de trabalho


In [ ]:
df_prof = df.filter(
    (F.col("senioridade") != "nao_informado") &
    (F.col("cargo") != "nao_informado")
)
print("Profissionais de dados (com cargo/senioridade informados):", df_prof.count())

In [ ]:
df_gestor = df.filter(F.col("cargo_gestor") != "nao_informado")
print("Gestores/líderes (respondem estratégia de IA da empresa):", df_gestor.count())

def save(pdf, name):
    path = os.path.join(OUT, name)
    pdf.to_csv(path, index=False)
    print("salvo:", path)

A base analisada contém 14.005 respondentes da pesquisa State of Data Brazil (ciclo 2025/2026), reunindo perfil demográfico, cargo, senioridade, remuneração, modelo de trabalho, stack tecnológico e adoção de Inteligência Artificial Generativa. O processamento e as agregações foram realizados com Apache Spark (PySpark/Spark SQL), o que permitiu tratar o volume de colunas (64) e registros de forma escalável antes da geração dos gráficos.
A pesquisa possui dois blocos de respondentes com propósitos diferentes, tratados separadamente na análise:

**- Profissionais individuais:** 10.176 registros de profissionais de dados (com

cargo e senioridade preenchidos) — usados para comparar região, senioridade e modelo de trabalho.

**-Gestores e lideranças:** 2.668 registros de gestores/líderes (gerentes, coordenadores, C-level) que responderam sobre a estratégia de dados/IA da própria empresa — usados para mapear prioridades, barreiras e maturidade organizacional.


In [ ]:
from pyspark.sql import functions as F

# 0. Garante que o df_prof seja recriado a partir do DataFrame original (df)
df_prof = df

# 1. Exibir Distribuição por Região (sem 'nao_informado')
df_prof.filter(
    (F.col("regiao") != "nao_informado") & F.col("regiao").isNotNull()
).groupBy("regiao").count().orderBy(F.desc("count")).show(truncate=False)

# 2. Exibir Distribuição por Senioridade (sem 'nao_informado')
df_prof.filter(
    (F.col("senioridade") != "nao_informado")
    & F.col("senioridade").isNotNull()
).groupBy("senioridade").count().orderBy(F.desc("count")).show(truncate=False)

# 3. Exibir Distribuição por Forma de Trabalho (sem 'nao_informado')
df_prof.filter(
    (F.col("forma_trabalho") != "nao_informado")
    & F.col("forma_trabalho").isNotNull()
).groupBy("forma_trabalho").count().orderBy(F.desc("count")).show(
    truncate=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="white")

# 1. Preparação dos DataFrames em Pandas (FILTRANDO 'nao_informado' E 'null')
df_regiao = (
    df_prof.filter(
        (F.col("regiao") != "nao_informado") & F.col("regiao").isNotNull()
    )
    .groupBy("regiao")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

df_senioridade = (
    df_prof.filter(
        (F.col("senioridade") != "nao_informado")
        & F.col("senioridade").isNotNull()
    )
    .groupBy("senioridade")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

df_trabalho = (
    df_prof.filter(
        (F.col("forma_trabalho") != "nao_informado")
        & F.col("forma_trabalho").isNotNull()
    )
    .groupBy("forma_trabalho")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

# 2. Criando a figura e os eixos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# GRÁFICO 1: Região
sns.barplot(
    data=df_regiao,
    x="regiao",
    y="count",
    hue="regiao",
    palette="viridis",
    legend=False,
    ax=axes[0],
)

axes[0].set_title(
    "Distribuição por Região",
    fontsize=12,
    fontweight="bold",
    pad=12
)
axes[0].set_xlabel("Região")
axes[0].set_ylabel("")
axes[0].set_yticks([])
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_ylim(0, df_regiao["count"].max() * 1.15)

for c in axes[0].containers:
    axes[0].bar_label(
        c,
        fmt="%d",
        padding=3,
        fontweight="bold"
    )


# GRÁFICO 2: Senioridade
sns.barplot(
    data=df_senioridade,
    x="senioridade",
    y="count",
    hue="senioridade",
    palette="viridis",
    legend=False,
    ax=axes[1],
)

axes[1].set_title(
    "Distribuição por Senioridade",
    fontsize=12,
    fontweight="bold",
    pad=12
)
axes[1].set_xlabel("Senioridade")
axes[1].set_ylabel("")
axes[1].set_yticks([])
axes[1].tick_params(axis="x", rotation=30)
axes[1].set_ylim(0, df_senioridade["count"].max() * 1.15)

for c in axes[1].containers:
    axes[1].bar_label(
        c,
        fmt="%d",
        padding=3,
        fontweight="bold"
    )


# GRÁFICO 3: Forma de Trabalho
sns.barplot(
    data=df_trabalho,
    x="forma_trabalho",
    y="count",
    hue="forma_trabalho",
    palette="viridis",
    legend=False,
    ax=axes[2],
)

axes[2].set_title(
    "Distribuição por Forma de Trabalho",
    fontsize=12,
    fontweight="bold",
    pad=12
)
axes[2].set_xlabel("Forma de Trabalho")
axes[2].set_ylabel("")
axes[2].set_yticks([])
axes[2].tick_params(axis="x", rotation=15)
axes[2].set_ylim(0, df_trabalho["count"].max() * 1.15)

for c in axes[2].containers:
    axes[2].bar_label(
        c,
        fmt="%d",
        padding=3,
        fontweight="bold"
    )


# Remove bordas desnecessárias
sns.despine(left=True)

plt.tight_layout()
plt.show()

A concentração é acentuada: 6 em cada 10 profissionais de dados do país estão no Sudeste, e Sudeste + Sul somam quase 80% da amostra. Norte e Centro-Oeste, juntos, não chegam a 8% do total — um desequilíbrio muito maior do que a distribuição populacional do país, o que indica escassez relativa de talento (e de empresas maduras em dados) fora do eixo Sul-Sudeste.

##Salário - Região, Senioridade e Forma de Trabalho

In [ ]:
from pyspark.sql import functions as F

# 0. Criando a variável df_sal necessária para os cálculos
df_sal = (
    df_prof.filter(
        (F.col("faixa_salarial_ordem") != "99")
        & (F.col("faixa_salarial_ordem") != "nao_informado")
        & F.col("faixa_salarial_ordem").isNotNull()
    )
    # Converte para inteiro para permitir a média estatística
    .withColumn(
        "faixa_salarial_ordem", F.col("faixa_salarial_ordem").cast("int")
    )
)

# 1. Salário por Região (sem 'nao_informado')
df_sal.filter(
    (F.col("regiao") != "nao_informado") & F.col("regiao").isNotNull()
).groupBy("regiao").agg(
    F.round(F.avg("faixa_salarial_ordem"), 2).alias("media_ordem_salarial"),
    F.count("*").alias("n"),
).orderBy(
    F.desc("media_ordem_salarial")
).show(
    truncate=False
)

# 2. Salário por Senioridade (sem 'nao_informado')
df_sal.filter(
    (F.col("senioridade") != "nao_informado")
    & F.col("senioridade").isNotNull()
).groupBy("senioridade").agg(
    F.round(F.avg("faixa_salarial_ordem"), 2).alias("media_ordem_salarial"),
    F.count("*").alias("n"),
).orderBy(
    F.desc("media_ordem_salarial")
).show(
    truncate=False
)

# 3. Salário por Forma de Trabalho (sem 'nao_informado')
df_sal.filter(
    (F.col("forma_trabalho") != "nao_informado")
    & F.col("forma_trabalho").isNotNull()
).groupBy("forma_trabalho").agg(
    F.round(F.avg("faixa_salarial_ordem"), 2).alias("media_ordem_salarial"),
    F.count("*").alias("n"),
).orderBy(
    F.desc("media_ordem_salarial")
).show(
    truncate=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="white")

# 1. Preparação dos DataFrames de Média Salarial
# (Com Filtros de 'nao_informado' e '99')
df_sal_base = df_prof.filter(
    (F.col("faixa_salarial_ordem") != "99")
    & F.col("faixa_salarial_ordem").isNotNull()
).withColumn(
    "faixa_salarial_ordem",
    F.col("faixa_salarial_ordem").cast("int")
)

# Média Salarial por Região
df_sal_regiao = (
    df_sal_base.filter(
        (F.col("regiao") != "nao_informado")
        & F.col("regiao").isNotNull()
    )
    .groupBy("regiao")
    .agg(
        F.round(
            F.avg("faixa_salarial_ordem"), 2
        ).alias("media_ordem_salarial")
    )
    .orderBy(F.desc("media_ordem_salarial"))
    .toPandas()
)

# Média Salarial por Senioridade
df_sal_senioridade = (
    df_sal_base.filter(
        (F.col("senioridade") != "nao_informado")
        & F.col("senioridade").isNotNull()
    )
    .groupBy("senioridade")
    .agg(
        F.round(
            F.avg("faixa_salarial_ordem"), 2
        ).alias("media_ordem_salarial")
    )
    .orderBy(F.desc("media_ordem_salarial"))
    .toPandas()
)

# Média Salarial por Forma de Trabalho
df_sal_trabalho = (
    df_sal_base.filter(
        (F.col("forma_trabalho") != "nao_informado")
        & F.col("forma_trabalho").isNotNull()
    )
    .groupBy("forma_trabalho")
    .agg(
        F.round(
            F.avg("faixa_salarial_ordem"), 2
        ).alias("media_ordem_salarial")
    )
    .orderBy(F.desc("media_ordem_salarial"))
    .toPandas()
)

# 2. Criando o painel de 3 gráficos lado a lado
fig, axes = plt.subplots(1, 3, figsize=(18, 5))


# --- GRÁFICO 1: Média Salarial por Região ---
sns.barplot(
    data=df_sal_regiao,
    x="regiao",
    y="media_ordem_salarial",
    hue="regiao",
    palette="viridis",
    legend=False,
    ax=axes[0],
)

axes[0].set_title(
    "Média Salarial por Região",
    fontsize=12,
    fontweight="bold",
    pad=12
)
axes[0].set_xlabel("Região")
axes[0].set_ylabel("")
axes[0].set_yticks([])
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_ylim(
    0,
    df_sal_regiao["media_ordem_salarial"].max() * 1.15
)

for c in axes[0].containers:
    axes[0].bar_label(
        c,
        fmt="%.2f",
        padding=3,
        fontweight="bold"
    )


# --- GRÁFICO 2: Média Salarial por Senioridade ---
sns.barplot(
    data=df_sal_senioridade,
    x="senioridade",
    y="media_ordem_salarial",
    hue="senioridade",
    palette="viridis",
    legend=False,
    ax=axes[1],
)

axes[1].set_title(
    "Média Salarial por Senioridade",
    fontsize=12,
    fontweight="bold",
    pad=12
)
axes[1].set_xlabel("Senioridade")
axes[1].set_ylabel("")
axes[1].set_yticks([])
axes[1].tick_params(axis="x", rotation=30)
axes[1].set_ylim(
    0,
    df_sal_senioridade["media_ordem_salarial"].max() * 1.15
)

for c in axes[1].containers:
    axes[1].bar_label(
        c,
        fmt="%.2f",
        padding=3,
        fontweight="bold"
    )


# --- GRÁFICO 3: Média Salarial por Forma de Trabalho ---
sns.barplot(
    data=df_sal_trabalho,
    x="forma_trabalho",
    y="media_ordem_salarial",
    hue="forma_trabalho",
    palette="viridis",
    legend=False,
    ax=axes[2],
)

axes[2].set_title(
    "Média Salarial por Forma de Trabalho",
    fontsize=12,
    fontweight="bold",
    pad=12
)
axes[2].set_xlabel("Forma de Trabalho")
axes[2].set_ylabel("")
axes[2].set_yticks([])
axes[2].tick_params(axis="x", rotation=15)
axes[2].set_ylim(
    0,
    df_sal_trabalho["media_ordem_salarial"].max() * 1.15
)

for c in axes[2].containers:
    axes[2].bar_label(
        c,
        fmt="%.2f",
        padding=3,
        fontweight="bold"
    )


# Remove bordas para acabamento limpo
sns.despine(left=True)

plt.tight_layout()
plt.show()

**- Região:** a diferença entre regiões existe, mas é a menos acentuada dos três recortes — o índice varia de 5,79 (Nordeste) a 6,43 (Sudeste), uma diferença de cerca de 11%. Ou seja, a disparidade regional aparece muito mais na quantidade de vagas/profissionais do que no salário de quem já está empregado.

**- Senioridade:** é o fator com maior impacto salarial. O índice médio sobe de 4,0 (júnior) para 6,0 (pleno), 7,9 (sênior) e 9,2 (especialista), especialistas ganham, em média, mais que o dobro dos juniores.

**- Modelo de trabalho:** há um prêmio salarial claro para o trabalho remoto (índice 6,86) frente ao presencial (4,86), uma diferença de mais de 40%. Isso é coerente com o fato de o remoto abrir acesso a vagas de empresas de outras regiões (inclusive do exterior) e a perfis mais seniores.


#Resposta 7 - Quais oportunidades e desafios podem ser identificados para empresas que desejam investir em Dados e Inteligência Artificial?

##Adoção de IA

In [ ]:
import pyspark.sql.functions as F

def taxa_uso_ia(col):
    return (
        df_prof.groupBy(col)
        .agg(
            F.count("*").alias("n"),
            F.sum(F.when(F.col("usa_ia") == "sim", 1).otherwise(0)).alias("usa_ia_sim")
        )
        .withColumn("perc_usa_ia", F.round(F.col("usa_ia_sim") * 100.0 / F.col("n"), 1))
        .orderBy(F.desc("perc_usa_ia"))
        .toPandas()
    )

# Exibindo os resultados diretamente na tela
print("--- Uso de IA por Região ---")
print(taxa_uso_ia("regiao"))
print("\n") # Pula uma linha para facilitar a leitura

print("--- Uso de IA por Senioridade ---")
print(taxa_uso_ia("senioridade"))
print("\n")

print("--- Uso de IA por Forma de Trabalho ---")
print(taxa_uso_ia("forma_trabalho"))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Tirando os nao informados dos ultimos dois graficos

import pyspark.sql.functions as F

def taxa_uso_ia(col):
    return (
        df_prof
        # Exclui valores nulos e a categoria 'nao_informado'
        .filter((F.col(col).isNotNull()) & (F.col(col) != "nao_informado"))
        .groupBy(col)
        .agg(
            F.count("*").alias("n"),
            F.sum(F.when(F.col("usa_ia") == "sim", 1).otherwise(0)).alias("usa_ia_sim")
        )
        .withColumn("perc_usa_ia", F.round(F.col("usa_ia_sim") * 100.0 / F.col("n"), 1))
        .orderBy(F.desc("perc_usa_ia"))
        .toPandas()
    )

    #-

def plotar_painel_minimalista_uso_ia(df_regiao, df_senioridade, df_forma_trabalho):
    # Aplica o estilo limpo sem grade de fundo
    sns.set_theme(style="white")

    # Criamos 1 linha e 3 colunas de gráficos lado a lado
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Mapeamento dos dados e títulos para iterar e gerar os gráficos
    dados_graficos = [
        (df_regiao, "regiao", "Taxa de Uso de IA por Região", axes[0]),
        (df_senioridade, "senioridade", "Taxa de Uso de IA por Senioridade", axes[1]),
        (df_forma_trabalho, "forma_trabalho", "Taxa de Uso de IA por Forma de Trabalho", axes[2])
    ]

    for df, col, titulo, ax in dados_graficos:
        sns.barplot(
            data=df,
            x="perc_usa_ia",
            y=col,
            hue=col,
            palette="viridis",
            ax=ax
        )

        # Remove legenda redundante
        if ax.get_legend():
            ax.get_legend().remove()

        # Adiciona os números percentuais fora de cada barra
        for p in ax.patches:
            largura = p.get_width()
            if largura > 0:
                ax.annotate(
                    f"{largura:.1f}%",
                    (largura + 2, p.get_y() + p.get_height() / 2.0),
                    ha="left",
                    va="center",
                    fontsize=10,
                    fontweight="bold"
                )

        #AJUSTES MINIMALISTAS
        ax.set_xticks([])         # Oculta números do eixo X
        ax.set_xlabel("")         # Oculta título do eixo X
        ax.set_ylabel("")         # Oculta título do eixo Y
        ax.set_xlim(0, 115)       # Espaço extra para o texto da %
        ax.set_title(titulo, fontsize=12, fontweight="bold", pad=15)

        # Remove todas as bordas (caixas do gráfico)
        sns.despine(ax=ax, top=True, right=True, left=True, bottom=True)

    plt.tight_layout()

    # Se estiver no Databricks, utilize: display(fig)
    plt.show()

# Execução única para os 3 DataFrames
df_regiao = taxa_uso_ia("regiao")
df_senioridade = taxa_uso_ia("senioridade")
df_forma_trabalho = taxa_uso_ia("forma_trabalho")

plotar_painel_minimalista_uso_ia(df_regiao, df_senioridade, df_forma_trabalho)



A adoção de IA já é alta em todos os grupos (75% a 87%), mas com gradientes visíveis: por região, Centro-Oeste lidera (86,8%) e Sul aparece um pouco abaixo (82,6%) — diferença pequena, sugerindo que o uso individual de IA já se popularizou de forma bastante homogênea pelo país. Por senioridade, o uso cresce com a experiência (79,2% no júnior até 86,2% no especialista), o que pode refletir tanto maior familiaridade quanto mais autonomia para adotar novas ferramentas. Por modelo de trabalho, remoto e híbrido flexível lideram (85%) enquanto o presencial fica visivelmente atrás (75,5%), uma diferença de quase 10 pontos percentuais.


###Prioridade de IA nas empresas - Gestores/lideres respondentes

In [ ]:
import pyspark.sql.functions as F

# 10. Prioridade de IA Geral
df_prioridade_geral = (
    df_gestor.filter(F.col("ia_prioridade") != "nao_informado")
    .groupBy("ia_prioridade")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

print("--- Prioridade de IA (Geral) ---")
print(df_prioridade_geral)
print("\n")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calcula o percentual sobre o total
total = df_prioridade_geral["count"].sum()
df_prioridade_geral["perc"] = (df_prioridade_geral["count"] / total) * 100

# 2. Configuração do gráfico minimalista
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(7, 5))

# Invertemos X e Y para gerar colunas verticais
sns.barplot(
    data=df_prioridade_geral,
    x="ia_prioridade",
    y="perc",
    hue="ia_prioridade",
    palette="viridis",
    ax=ax
)

# Remove legenda redundante
if ax.get_legend():
    ax.get_legend().remove()

# Adiciona os números em PORCENTAGEM ACIMA das colunas
for p in ax.patches:
    altura = p.get_height()
    if altura > 0:
        ax.annotate(
            f"{altura:.1f}%",
            (p.get_x() + p.get_width() / 2.0, altura + 1.5), # Posição centralizada no topo da coluna
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )

# --- AJUSTES MINIMALISTAS ---
ax.set_yticks([])         # Remove os números do eixo Y
ax.set_xlabel("")         # Oculta título do eixo X
ax.set_ylabel("")         # Oculta título do eixo Y
ax.set_ylim(0, df_prioridade_geral["perc"].max() * 1.2) # Espaço no topo para não cortar a %

# Remove todas as bordas
sns.despine(ax=ax, top=True, right=True, left=True, bottom=True)

# Título do gráfico
ax.set_title("Prioridade de IA (Geral)", fontsize=13, fontweight="bold", pad=20)

plt.tight_layout()

# Se estiver no Databricks, use: display(fig)
plt.show()

Quase metade dos gestores (49%) afirma que IA é hoje foco principal ou prioridade estratégica da empresa, e outros 29% dizem que é ao menos uma prioridade parcial, ou seja, quase 8 em cada 10 empresas já têm algum nível de investimento em curso. Apenas 19% declaram que IA não é prioridade. Esse cenário indica uma janela de oportunidade real para fornecedores de tecnologia, consultorias e plataformas de dados: a demanda já existe e tende a crescer, não precisa ser 'criada'.

###Motivo de não adoção de IA

In [ ]:
import pandas as pd
import pyspark.sql.functions as F

# 1. Dicionário com o mapeamento dos nomes amigáveis
mapeamento_motivos = {
    "motivo_expertise": "Falta de expertise interna",
    "motivo_casos_uso": "Falta de casos de uso",
    "motivo_dados_prontos": "Dados não prontos para IA",
    "motivo_roi": "ROI não aprovado",
    "motivo_seguranca": "Segurança e privacidade"
}

rows = []
# 2. Iteramos pelas chaves do dicionário
for coluna, nome_amigavel in mapeamento_motivos.items():
    n_sim = df_gestor.filter(F.col(coluna) == "sim").count()
    # Adicionamos o nome amigável em vez do código técnico
    rows.append((nome_amigavel, n_sim))

# 3. Cria o DataFrame e ordena
pdf_motivos = (
    pd.DataFrame(rows, columns=["motivo", "n_sim"])
    .sort_values("n_sim", ascending=False)
    .reset_index(drop=True)
)

print("--- Motivos para não adotar IA ---")
print(pdf_motivos)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calcula o total de gestores e cria a porcentagem para cada motivo
total_gestores = df_gestor.count()
pdf_motivos["perc"] = (pdf_motivos["n_sim"] / total_gestores) * 100

# 2. Configura o gráfico minimalista em colunas verticais
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=pdf_motivos,
    x="motivo",
    y="perc",
    hue="motivo",
    palette="viridis",
    ax=ax
)

# Remove a legenda redundante
if ax.get_legend():
    ax.get_legend().remove()

# Adiciona o número em PORCENTAGEM ACIMA de cada coluna
for p in ax.patches:
    altura = p.get_height()
    if altura > 0:
        ax.annotate(
            f"{altura:.1f}%",
            (p.get_x() + p.get_width() / 2.0, altura + 1.2),
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold"
        )

# --- AJUSTES MINIMALISTAS ---
ax.set_yticks([])         # Esconde valores do eixo Y
ax.set_xlabel("")         # Remove título do eixo X
ax.set_ylabel("")         # Remove título do eixo Y
ax.set_ylim(0, pdf_motivos["perc"].max() * 1.2) # Garante espaço no topo

# Formata os rótulos do eixo X (substitui underline por espaço e deixa em caixa alta/capitalizado)
labels = [item.get_text().replace("_", " ").title() for item in ax.get_xticklabels()]
ax.set_xticklabels(labels, fontsize=10)

# Remove todas as bordas
sns.despine(ax=ax, top=True, right=True, left=True, bottom=True)

# Título do gráfico
ax.set_title("Motivos para Não Adoção de IA (%)", fontsize=13, fontweight="bold", pad=20)

plt.tight_layout()
plt.show()

Entre os gestores que apontam barreiras, os motivos mais citados são falta de expertise interna (31,6%) e falta de casos de uso claros (30,9%), seguidos de perto por dados da empresa não estarem prontos para uso em IA (28,4%) e preocupações com segurança e privacidade (27,7%). ROI não comprovado aparece em último lugar (20,3%) — sinal de que a barreira não é a falta de crença no retorno financeiro, e sim a falta de capacitação, direção clara de uso e qualidade/governança dos dados.

###Setores que mais priorizam a IA - Gestores/lideres

In [ ]:
import pyspark.sql.functions as F

# Processamento dos dados
df_setores_prioridade = (
    df_gestor.filter(F.col("setor") != "nao_informado")
    .groupBy("setor").agg(
        F.count("*").alias("n"),
        F.sum(F.when(F.col("ia_prioridade").isin("prioritaria", "foco_principal"), 1).otherwise(0)).alias("prioriza_ia")
    )
    .withColumn("perc_prioriza_ia", F.round(F.col("prioriza_ia") * 100.0 / F.col("n"), 1))
    .filter(F.col("n") >= 15)
    .orderBy(F.desc("perc_prioriza_ia"))
    .toPandas()
)

# Exibição do resultado em tela
print("--- Setores com Maior Prioridade em IA ---")
print(df_setores_prioridade)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configura o tema sem grade
sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(9, 5))

# Gráfico de barras horizontais
sns.barplot(
    data=df_setores_prioridade,
    x="perc_prioriza_ia",
    y="setor",
    hue="setor",
    palette="viridis",
    ax=ax
)

# Remove legenda redundante
if ax.get_legend():
    ax.get_legend().remove()

# Adiciona a porcentagem na ponta de cada barra
for p in ax.patches:
    largura = p.get_width()
    if largura > 0:
        ax.annotate(
            f"{largura:.1f}%",
            (largura + 1.5, p.get_y() + p.get_height() / 2.0),
            ha="left",
            va="center",
            fontsize=10,
            fontweight="bold"
        )

# --- AJUSTES MINIMALISTAS ---
ax.set_xticks([])         # Oculta números do eixo X
ax.set_xlabel("")         # Remove título do eixo X
ax.set_ylabel("")         # Remove título do eixo Y
ax.set_xlim(0, 115)       # Espaço para o rótulo % não cortar

# Remove todas as bordas do gráfico
sns.despine(ax=ax, top=True, right=True, left=True, bottom=True)

# Título do gráfico
ax.set_title("Setores que Priorizam IA", fontsize=13, fontweight="bold", pad=20)

plt.tight_layout()

# Se estiver no Databricks, use: display(fig)
plt.show()

Tecnologia/fábricas de software (60,7%), Internet/E-commerce (59,3%), Consultoria (53,5%) e Finanças/Bancos (50,5%) são os setores onde IA já é prioridade para a maioria das empresas, bons pontos de entrada para quem vende soluções de dados/IA. No outro extremo, Agronegócio (20,5%) e Setor Público (25,8%) ainda priorizam pouco, o que pode representar tanto um risco de baixa demanda no curto prazo quanto uma oportunidade de mercado ainda pouco explorado por concorrentes.

### **Conclusão**
O mercado brasileiro de Dados e IA é ao mesmo tempo maduro e desigual. Maduro porque a adoção de IA generativa já é alta entre profissionais (75%+ em todos os grupos) e quase metade das empresas já trata o tema como prioridade estratégica, com uma base tecnológica (cloud, lakehouse, BI) já consolidada. Desigual porque essa maturidade está fortemente concentrada no eixo Sudeste-Sul, cresce com a senioridade dos profissionais e é favorecida pelo modelo remoto de trabalho.
Para empresas que avaliam investir em Dados e IA, a oportunidade mais clara está em atacar as barreiras de capacitação e qualidade de dados — não o convencimento sobre o valor da IA, que já existe — e em considerar regiões e setores ainda pouco maduros como frentes de expansão de médio prazo, ao mesmo tempo em que se compete por talento sênior e especializado, hoje escasso e concentrado geograficamente.
